# 03 — Claude Code, MCP & Integration

This live lab uses Claude to review the design decisions taught in the Claude Code and MCP module: permissions, durable context, reusable workflow surfaces, transport, scope, and enterprise identity. Start with the [22-screen module index](../course%20content%20HTML/03-claude-code-mcp-integration/index.html).

> Running all cells requires `OPENROUTER_API_KEY` and uses paid API tokens. The notebook does not execute Claude Code or connect to a real enterprise system.

## Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_support.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The key is loaded from the root `.env`; the model name is an explicit, pinned default.

In [ ]:
from study_support import claude_client, claude_model, message_text

client = claude_client()
MODEL = claude_model()

## Permission mode follows worst-case cost

Choose permissions from the risk of a wrong action, not the desire to remove prompts. Place the human gate immediately before an irreversible or production-impacting step. [Course screen 2](../course%20content%20HTML/03-claude-code-mcp-integration/02-permission-modes-human-gates.html#screen-02--claude-code-agent-loop-permission-modes-settings-and-where-a-human-gate-goes)

In [ ]:
scenario = {
    "task": "inspect a migration and deploy it to production",
    "permission_mode": "acceptEdits",
    "production_gate": "human approval",
}

Ask Claude for a review, but keep the allowed modes in trusted instructions rather than inside the scenario.

In [ ]:
import json

permission_review = client.messages.create(
    model=MODEL,
    max_tokens=180,
    system="Review permissions. Prefer least privilege and require approval before production.",
    messages=[{"role": "user", "content": json.dumps(scenario)}],
)
print(message_text(permission_review))

## Put durable guidance in the right mechanism

`CLAUDE.md` holds short project-wide facts. Rules scope instructions to matching paths. Hooks enforce deterministic checks. Subagents isolate delegated context. [Course screen 5](../course%20content%20HTML/03-claude-code-mcp-integration/03-durable-project-context.html#screen-05--durable-project-context-with-claudemd-rules-files-hooks-and-subagents)

In [ ]:
project_context = """
Project: payments API
Commands: pytest -q
Constraint: never deploy without human approval
""".strip()

The Messages API keeps the system prompt separate from the user/assistant conversation. That makes the durable instruction boundary explicit.

In [ ]:
context_check = client.messages.create(
    model=MODEL,
    max_tokens=100,
    system=project_context,
    messages=[{
        "role": "user",
        "content": "What check must happen immediately before deployment?",
    }],
)
print(message_text(context_check))

## Choose the smallest reusable surface

Use a built-in for native capability, a custom tool for one application-owned action, a Skill for a reusable workflow, and MCP for a capability shared and maintained independently. [Course screens 8 and 12](../course%20content%20HTML/03-claude-code-mcp-integration/04-packaging-workflows.html#screen-08--packaging-a-workflow-as-a-plugin-skills-custom-commands-and-marketplace-install)

In [ ]:
surface_case = "Several applications need the same maintained customer-record lookup."
surface_prompt = (
    "Choose exactly one: built-in, custom tool, Skill, or MCP. "
    f"Scenario: {surface_case}"
)

The output is advisory. Architecture remains an application decision.

In [ ]:
surface_answer = client.messages.create(
    model=MODEL,
    max_tokens=20,
    messages=[{"role": "user", "content": surface_prompt}],
)
print(message_text(surface_answer))

## MCP separates capability from the model call

MCP uses JSON-RPC. Servers may expose tools (actions), resources (readable data), and prompts (reusable templates). The request below is data a client would send to a server; Claude does not execute it directly. [Course screen 12](../course%20content%20HTML/03-claude-code-mcp-integration/05-mcp-servers.html#screen-12--building-and-configuring-an-mcp-server-transport-scope-and-the-github-server)

In [ ]:
mcp_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": "get_customer", "arguments": {"customer_id": "C-7"}},
}
assert mcp_request["method"] == "tools/call"

Transport and scope are independent. Use `stdio` for a local child process and HTTP for a remote service; scope credentials as narrowly as the intended users and environment.

In [ ]:
mcp_config = {
    "transport": "HTTP",
    "scope": "project",
    "credential": "$CUSTOMER_MCP_TOKEN",
    "allowed_tools": ["get_customer"],
}
assert mcp_config["credential"].startswith("$")

## Review the enterprise boundary

Enterprise integration starts with identity, secret storage, rotation, authorization, and auditability—not merely a successful connection. [Course screen 15](../course%20content%20HTML/03-claude-code-mcp-integration/06-enterprise-integration.html#screen-15--connecting-claude-to-enterprise-systems-and-authenticating-it-securely)

In [ ]:
connection = {
    "identity": "svc-customer-readonly",
    "secret_source": "environment variable",
    "permissions": ["customer:read"],
    "audit_log": True,
    "rotation_owner": "platform-team",
}

Send configuration facts, never the credential value itself.

In [ ]:
security_review = client.messages.create(
    model=MODEL,
    max_tokens=160,
    system="Find missing integration controls. Be concise and do not invent facts.",
    messages=[{"role": "user", "content": json.dumps(connection)}],
)
print(message_text(security_review))

## Try it

Change the MCP scenario from a shared remote service to a developer-only local filesystem tool. Re-evaluate transport, scope, identity, and the location of the human gate.